# waste heat recovery cement retrofit NPV simulation

Run the waste heat recovery cement retrofit Monte Carlo simulation and visualize the resulting NPV distribution.

The summary also reports how many simulations have non-negative NPV (NPV >= 0) and how many have negative NPV.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cement.cement_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_RETROFIT_BAU_MODE,
    DEFAULT_SAMPLE_SIZE,
    simulate_cement_results,
)

from npv_summary import summarize_metric_signs


In [2]:
TECHNOLOGY = 'waste_heat_recovery'
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED
RETROFIT_BAU_MODE = DEFAULT_RETROFIT_BAU_MODE

results_by_technology = simulate_cement_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=(TECHNOLOGY,),
    retrofit_bau_mode=RETROFIT_BAU_MODE,
)
simulation = results_by_technology[TECHNOLOGY]
results = pd.DataFrame(simulation)
results.head()


,run_id,technology,technology_type,retrofit_bau_mode,annual_output_t,lifetime_years,capex_eur_per_t,fixed_opex_eur_per_t,variable_opex_eur_per_t,fuel_consumption_mwh_th_per_t,...,bau_variable_opex_eur_per_t,bau_fuel_consumption_mwh_th_per_t,bau_electricity_consumption_mwh_per_t,bau_emissions_tco2_per_t,capex_change_eur_per_t,fixed_opex_change_eur_per_t,variable_opex_change_eur_per_t,fuel_consumption_reduction_fraction,electricity_consumption_reduction_fraction,emissions_reduction_fraction
0,0,waste_heat_recovery,retrofit,sampled,1000000.0,25.0,169.276592,15.075239,5.135369,0.657332,...,5.135369,0.657332,0.084504,0.620035,3.797471,0.137330,0.0,0.0,0.239459,0.0
1,1,waste_heat_recovery,retrofit,sampled,1000000.0,25.0,163.165747,15.419946,5.125930,0.649577,...,5.125930,0.649577,0.093378,0.602551,4.388179,0.473028,0.0,0.0,0.264979,0.0
2,2,waste_heat_recovery,retrofit,sampled,1000000.0,25.0,169.594828,14.892180,5.431670,0.697878,...,5.431670,0.697878,0.085678,0.683904,2.422870,0.427930,0.0,0.0,0.238513,0.0
3,3,waste_heat_recovery,retrofit,sampled,1000000.0,25.0,178.023906,14.235324,5.260010,0.693428,...,5.260010,0.693428,0.089539,0.634547,14.076546,0.233596,0.0,0.0,0.380512,0.0
4,4,waste_heat_recovery,retrofit,sampled,1000000.0,25.0,160.612624,15.145209,5.371749,0.694798,...,5.371749,0.694798,0.084480,0.652753,8.729077,0.425531,0.0,0.0,0.244413,0.0


In [3]:
npv_million_eur = results["npv_eur"] / 1_000_000
lcoc_eur_per_t = results["lcoc_eur_per_t"]
levelized_net_margin_eur_per_t = results["levelized_net_margin_eur_per_t"]

summary = pd.concat(
    [
        npv_million_eur.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "NPV million EUR"
        ),
        lcoc_eur_per_t.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "LCOC EUR/t cement"
        ),
        levelized_net_margin_eur_per_t.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "Levelized net margin EUR/t cement"
        ),
    ],
    axis=1,
)

npv_signs = summarize_metric_signs(npv_million_eur)
npv_sign_summary = pd.DataFrame(
    {
        "NPV category": ["Non-negative (NPV >= 0)", "Negative (NPV < 0)"],
        "Simulation count": [
            npv_signs["non_negative_count"],
            npv_signs["negative_count"],
        ],
        "Simulation share": [
            npv_signs["non_negative_share"],
            1.0 - npv_signs["non_negative_share"],
        ],
    }
)

display(summary)
display(npv_sign_summary)


,NPV million EUR,LCOC EUR/t cement,Levelized net margin EUR/t cement
count,100000.000000,100000.000000,100000.000000
mean,471.184009,105.860057,44.139943
std,39.085648,3.661496,3.661496
min,303.238510,93.135513,28.407013
5%,404.939114,100.013044,37.934202
50%,472.281923,105.757206,44.242794
95%,533.599570,112.065798,49.986956
max,607.015668,121.592987,56.864487


,NPV category,Simulation count,Simulation share
0,Non-negative (NPV >= 0),100000,1.0
1,Negative (NPV < 0),0,0.0


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(npv_million_eur, bins=50, color="tab:gray", edgecolor="white", alpha=0.8)
ax.axvline(npv_million_eur.mean(), color="tab:blue", linewidth=2, label="Mean")
ax.axvline(npv_million_eur.median(), color="tab:orange", linewidth=2, label="Median")
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"waste heat recovery cement retrofit NPV distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("NPV (million EUR)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42756/1321164233.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    levelized_net_margin_eur_per_t,
    bins=50,
    color="tab:green",
    edgecolor="white",
    alpha=0.8,
)
ax.axvline(
    levelized_net_margin_eur_per_t.mean(),
    color="tab:blue",
    linewidth=2,
    label="Mean",
)
ax.axvline(
    levelized_net_margin_eur_per_t.median(),
    color="tab:orange",
    linewidth=2,
    label="Median",
)
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"waste heat recovery cement retrofit levelized net margin distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("Levelized net margin (EUR/t cement)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42756/2372423438.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
annual_components = results[
    [
        "annual_revenue_eur",
        "annual_fixed_opex_eur",
        "annual_variable_opex_eur",
        "annual_fuel_cost_eur",
        "annual_electricity_cost_eur",
        "annual_emissions_cost_eur",
        "annual_net_cash_flow_eur",
    ]
] / 1_000_000

annual_components.mean().rename("Mean annual value, million EUR")


annual_revenue_eur             150.000000
annual_fixed_opex_eur           14.632301
annual_variable_opex_eur         5.167223
annual_fuel_cost_eur             8.074949
annual_electricity_cost_eur     11.389464
annual_emissions_cost_eur       50.670171
annual_net_cash_flow_eur        60.065893
Name: Mean annual value, million EUR, dtype: float64

In [7]:
retrofit_columns = [
    "capex_change_eur_per_t",
    "fixed_opex_change_eur_per_t",
    "variable_opex_change_eur_per_t",
    "fuel_consumption_reduction_fraction",
    "electricity_consumption_reduction_fraction",
    "emissions_reduction_fraction",
    "bau_capex_eur_per_t",
    "bau_fixed_opex_eur_per_t",
    "bau_variable_opex_eur_per_t",
    "bau_fuel_consumption_mwh_th_per_t",
    "bau_electricity_consumption_mwh_per_t",
    "bau_emissions_tco2_per_t",
]

available_retrofit_columns = [column for column in retrofit_columns if column in results]
retrofit_summary = results[available_retrofit_columns].describe(
    percentiles=[0.05, 0.5, 0.95]
)
retrofit_summary


,capex_change_eur_per_t,fixed_opex_change_eur_per_t,variable_opex_change_eur_per_t,fuel_consumption_reduction_fraction,electricity_consumption_reduction_fraction,emissions_reduction_fraction,bau_capex_eur_per_t,bau_fixed_opex_eur_per_t,bau_variable_opex_eur_per_t,bau_fuel_consumption_mwh_th_per_t,bau_electricity_consumption_mwh_per_t,bau_emissions_tco2_per_t
count,100000.000000,100000.000000,100000.0,100000.0,100000.000000,100000.0,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,9.993457,0.299893,0.0,0.0,0.284764,0.0,160.012499,14.332408,5.167223,0.666500,0.086635,0.633377
std,4.620559,0.115696,0.0,0.0,0.066328,0.0,5.770418,0.471200,0.235343,0.040049,0.004713,0.023522
min,2.000063,0.100013,0.0,0.0,0.170001,0.0,150.000263,13.009529,4.504187,0.610000,0.080000,0.600000
5%,2.803300,0.119735,0.0,0.0,0.181570,0.0,150.989862,13.451564,4.725440,0.614265,0.080499,0.602579
50%,9.975813,0.299488,0.0,0.0,0.284608,0.0,160.049191,14.411204,5.207742,0.659535,0.085806,0.629393
95%,17.202561,0.480021,0.0,0.0,0.388182,0.0,169.007981,14.949464,5.474133,0.741932,0.095517,0.677623
max,17.999974,0.500000,0.0,0.0,0.399999,0.0,169.999875,14.999998,5.499993,0.779352,0.099946,0.699866
